# 301 — MCP Server and Tools

Demonstrates the Phase 3 MCP tool layer. Covers:

1. Direct Python calls to all exposed tools (shared, graph, risk, trace)
2. How to start the MCP server for Claude Desktop or other MCP clients
3. The full list of exposed MCP tools and their contracts

**Start the server** (outside this notebook) with:
```bash
python -m src.mcp.server
```

In [ ]:
import sys
sys.path.insert(0, "..")

import json

from src.config import get_neo4j_settings
from src.storage.neo4j_repository import Neo4jRepository
from src.storage.trace_repository import TraceRepository
from src.tools.graph_tools import GraphTools
from src.tools.risk_tools import RiskTools
from src.tools.shared_tools import SharedTools
from src.tools.trace_tools import TraceTools

settings = get_neo4j_settings()
neo4j = Neo4jRepository(**vars(settings))
neo4j.test_connection()

graph   = GraphTools(neo4j)
risk    = RiskTools(neo4j)
trace   = TraceTools(TraceRepository(neo4j))
shared  = SharedTools(graph)

print("Tool layer ready")


def show(result):
    """Print a ToolResult summary and its structured data."""
    print(f"[{result.tool_name}] {'OK' if result.success else 'FAIL'} "
          f"({result.duration_ms} ms)")
    print(f"  {result.summary}")
    if result.data is not None:
        print(json.dumps(result.data, indent=2, default=str)[:1200])

## Shared tools

These orchestration-level tools support Phase 3 agent coordination.

In [ ]:
# resolve_entity — canonicalise a company name
result = shared.resolve_entity("VODAFONE")
show(result)

if result.success and result.data.get("resolved"):
    COMPANY = result.data["canonical_name"]
    print(f"\nCanonical name for next cells: {COMPANY!r}")
else:
    COMPANY = "VODAFONE GROUP PLC"  # fallback
    print(f"\nUsing fallback name: {COMPANY!r}")

In [ ]:
# validate_plan — check steps before handing to an orchestrator
plan = [
    {"step_id": "step_1", "tool_name": "company_profile",           "parameters": {"company_name": COMPANY}},
    {"step_id": "step_2", "tool_name": "expand_ownership",          "parameters": {"company_name": COMPANY}},
    {"step_id": "step_3", "tool_name": "ownership_complexity_check", "parameters": {"company_name": COMPANY}},
    {"step_id": "step_4", "tool_name": "address_risk_check",         "parameters": {"company_name": COMPANY}},
    {"step_id": "step_5", "tool_name": "industry_context_check",     "parameters": {"company_name": COMPANY}},
    {"step_id": "step_6", "tool_name": "evaluate_stop_conditions",   "parameters": {}},
]
show(shared.validate_plan(plan))

print()

# Invalid plan — unknown tool name
bad_plan = [
    {"step_id": "step_1", "tool_name": "nonexistent_tool"},
    {"step_id": "",       "tool_name": "entity_lookup"},
]
show(shared.validate_plan(bad_plan))

In [ ]:
# evaluate_stop_conditions — decision gate after gathering signals
findings_complete = {
    "ownership_complexity": {"risk_level": "MEDIUM"},
    "control_signals":       {"risk_level": "HIGH"},
    "address_risk":          {"risk_level": "LOW"},
    "industry_context":      {"risk_level": "MEDIUM"},
}
show(shared.evaluate_stop_conditions(findings_complete))

print()

# Incomplete — two signals missing
findings_partial = {
    "ownership_complexity": {"risk_level": "LOW"},
    "address_risk":         {"risk_level": "LOW"},
}
show(shared.evaluate_stop_conditions(findings_partial))

## Graph tools

In [ ]:
# expand_ownership — walk the ownership graph
show(graph.expand_ownership(COMPANY, max_depth=3))

In [ ]:
show(graph.company_profile(COMPANY))

## Risk tools

In [ ]:
for fn in [
    lambda: risk.ownership_complexity_check(COMPANY),
    lambda: risk.control_signal_check(COMPANY),
    lambda: risk.address_risk_check(COMPANY),
    lambda: risk.industry_context_check(COMPANY),
]:
    r = fn()
    print(f"[{r.tool_name}] risk_level={r.data.get('risk_level') if r.data else 'n/a'}  —  {r.summary}")

## Trace tools

Retrieval path: `list_recent_traces` → pick a `trace_id` → `retrieve_trace`.

In [ ]:
recent = trace.list_recent_traces(limit=5)
show(recent)

trace_id = None
if recent.success and recent.data:
    trace_id = recent.data[0]["trace_id"]
    print(f"\nLatest trace_id: {trace_id}")

In [ ]:
if trace_id:
    show(trace.retrieve_trace(trace_id))
else:
    print("No traces stored yet — run notebooks 207–210 first to generate trace data.")

In [ ]:
# find_traces_by_entity — entity-centric trace lookup
show(trace.find_traces_by_entity(COMPANY))

## MCP server

### Starting the server

```bash
# stdio transport (for Claude Desktop / MCP-compatible clients)
python -m src.mcp.server

# or via the MCP CLI
mcp run src/mcp/server.py

# inspect the tool manifest
mcp dev src/mcp/server.py
```

### Claude Desktop integration

Add to `~/.claude/claude_desktop_config.json`:

```json
{
  "mcpServers": {
    "entity-risk-ai": {
      "command": "python",
      "args": ["-m", "src.mcp.server"],
      "cwd": "/path/to/entity-risk-ai",
      "env": {
        "NEO4J_URI": "bolt://localhost:7687",
        "NEO4J_USERNAME": "neo4j",
        "NEO4J_PASSWORD": "your_password",
        "NEO4J_DATABASE": "neo4j"
      }
    }
  }
}
```

In [ ]:
# Inspect the registered MCP tools without starting the server
import sys
sys.path.insert(0, "..")

# Import the server module to access the registered tools
from src.mcp.server import mcp

print(f"Server name : {mcp.name}")
print()

# List all registered tools via the internal tool registry
tools = mcp._tool_manager._tools
print(f"Exposed MCP tools ({len(tools)}):")
print()
for name, tool in sorted(tools.items()):
    desc_first_line = (tool.description or "").splitlines()[0]
    print(f"  {name:<35}  {desc_first_line}")

In [ ]:
neo4j.close()
print("Driver closed")